# AQMN Spatial-Coverage Optimization -- Consolidated Notebook

Self-contained version of the pipeline in `src/analysis/`. All logic from
`config.py`, `io_utils.py`, `grid.py`, `coverage.py`, `optimize.py`,
`allocation.py`, `metrics.py`, `visualize.py`, `check_inputs.py`, and
`run_models.py` is inlined below -- no dependency on the `src` package, so
this notebook runs on its own. The original modules in `src/analysis/` are
unchanged and remain the source of truth for the CLI (`python -m
src.analysis.run_models`); this notebook is a convenience copy for
interactive / submission use.

**Run cells top to bottom.** Section order mirrors the module dependency
chain: config -> io_utils -> grid -> coverage -> optimize -> allocation ->
metrics -> visualize -> (optional) check_inputs -> run_models pipeline.

Outputs land in `data/results/n<N_NEW_STATIONS>/`, namespaced by station
count so re-running with a different `N_NEW_STATIONS` never overwrites a
previous run: per-model site tables (with lon/lat), per-model map PNGs, a
combined comparison PNG, and `model_comparison_n<N>.csv` (PCR + MARR).


## Imports

In [ ]:
import time
from pathlib import Path
from typing import Optional, Iterable

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.merge import merge
from shapely.geometry import Point
from sklearn.neighbors import BallTree
import pulp

import matplotlib
matplotlib.use("Agg")  # headless-safe backend; must be set before pyplot import
import matplotlib.pyplot as plt

try:
    from tqdm import tqdm
except ImportError:  # tqdm is optional -- falls back to a plain loop, no progress bar
    def tqdm(iterable, **kwargs):
        return iterable


## Config

Equivalent of `config.py`. `PROJECT_ROOT` assumes this notebook lives at the
project root (same level as `data/` and `src/`) -- adjust if you move it.


In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
GIS_DIR = DATA_DIR / "gis"

# Vector boundaries
ADMIN0_SHP = GIS_DIR / "phl_admin0.shp"
ADMIN2_SHP = GIS_DIR / "phl_admin2.shp"
ADMIN2_NAME_FIELD = "adm2_name"

# Population (1km density, ASCII XYZ -> loaded as CSV)
POP_DENSITY_CSV = GIS_DIR / "phl_pd_2020_1km_UNadj_ASCII_XYZ.csv"
POP_CSV_COLS = {"lon": "X", "lat": "Y", "density": "Z"}

# GHSL Built-up surface tiles (fraction built-up per cell, 3-arcsec ~ 100m, EPSG:4326)
GHSL_TILES = [
    GIS_DIR / "GHS_BUILT_S_E2020_GLOBE_R2023A_4326_3ss_V1_0_R7_C31.tif",
    GIS_DIR / "GHS_BUILT_S_E2020_GLOBE_R2023A_4326_3ss_V1_0_R8_C30.tif",
    GIS_DIR / "GHS_BUILT_S_E2020_GLOBE_R2023A_4326_3ss_V1_0_R8_C31.tif",
    GIS_DIR / "GHS_BUILT_S_E2020_GLOBE_R2023A_4326_3ss_V1_0_R9_C30.tif",
    GIS_DIR / "GHS_BUILT_S_E2020_GLOBE_R2023A_4326_3ss_V1_0_R9_C31.tif",
]
GHSL_MOSAIC_TIF = GIS_DIR / "ghsl_built_mosaic_phl.tif"  # cached merged output

# Stations
STATION_LIST_CSV = DATA_DIR / "station_list.csv"
STATION_LAT_COL = "lat"
STATION_LON_COL = "lon"

# Province-level population/area benchmark (for validation + min-station rule)
PROVINCE_STATS_CSV = DATA_DIR / "Population_LandArea_Density_Province.csv"

# Cached outputs -- kept separate from any pre-existing analytical_base_table.parquet
ABT_PARQUET = DATA_DIR / "gis_base_table.parquet"
RESULTS_DIR = DATA_DIR / "results"

# --- CRS -----------------------------------------------------------------
CRS_WGS84 = "EPSG:4326"
CRS_PROJECTED = "EPSG:3123"  # PRS92 / Philippines Zone III (meters)

# --- Model parameters (editable) -----------------------------------------
RADIUS_KM = 4.0                # station monitoring radius
CANDIDATE_CELL_KM = 2.0        # candidate grid resolution
N_NEW_STATIONS = 279           # total new stations to place
NO_OVERLAP_KM = 8.0            # min distance from an existing station (2x radius)
BUILTUP_THRESHOLD = 0.2        # min built-up fraction for a candidate to be "eligible"
MIN_STATIONS_PER_PROVINCE = 1  # floor used when splitting the budget across provinces
SOLVER_TIME_LIMIT_SEC = 300


## io_utils -- loaders for boundaries, population, built-up raster, stations

In [ ]:
def load_country_boundary() -> gpd.GeoDataFrame:
    gdf = gpd.read_file(ADMIN0_SHP)
    return gdf.to_crs(CRS_WGS84)


def load_provinces() -> gpd.GeoDataFrame:
    gdf = gpd.read_file(ADMIN2_SHP).to_crs(CRS_WGS84)
    if ADMIN2_NAME_FIELD not in gdf.columns:
        raise KeyError(
            f"'{ADMIN2_NAME_FIELD}' not found in phl_admin2 attributes.\n"
            f"Available columns: {list(gdf.columns)}\n"
            f"Update ADMIN2_NAME_FIELD in the Config cell above."
        )
    return gdf


def load_population_points() -> gpd.GeoDataFrame:
    """
    Loads the 1km population-density ASCII XYZ file and converts density -> counts.
    Returns a GeoDataFrame of points with columns:
        lon, lat, density, cell_area_km2, population, geometry
    """
    df = pd.read_csv(POP_DENSITY_CSV)
    lon_col = POP_CSV_COLS["lon"]
    lat_col = POP_CSV_COLS["lat"]
    dens_col = POP_CSV_COLS["density"]
    missing = [c for c in (lon_col, lat_col, dens_col) if c not in df.columns]
    if missing:
        raise KeyError(
            f"Column(s) {missing} not found in {POP_DENSITY_CSV.name}.\n"
            f"Available columns: {list(df.columns)}\n"
            f"Update POP_CSV_COLS in the Config cell above."
        )

    df = df.rename(columns={lon_col: "lon", lat_col: "lat", dens_col: "density"})
    df = df[df["density"] > 0].reset_index(drop=True)  # drop nodata / ocean cells

    # Per-row cell area in km^2 -- a "1km" lon/lat grid shrinks east-west as
    # |lat| grows, so area is NOT a flat 1 km^2 across the whole archipelago.
    lat_rad = np.radians(df["lat"].to_numpy())
    km_per_deg_lat = 111.32
    km_per_deg_lon = 111.32 * np.cos(lat_rad)
    cell_deg = 1.0 / km_per_deg_lat  # ~1km spacing expressed in degrees latitude
    df["cell_area_km2"] = (cell_deg * km_per_deg_lat) * (cell_deg * km_per_deg_lon)

    df["population"] = df["density"] * df["cell_area_km2"]

    gdf = gpd.GeoDataFrame(
        df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs=CRS_WGS84
    )
    return gdf


def build_ghsl_mosaic(force: bool = False) -> None:
    """Merges the GHSL built-up tiles once and caches the result to disk."""
    if GHSL_MOSAIC_TIF.exists() and not force:
        return
    srcs = [rasterio.open(p) for p in GHSL_TILES]
    mosaic, transform = merge(srcs)
    meta = srcs[0].meta.copy()
    meta.update(driver="GTiff", height=mosaic.shape[1], width=mosaic.shape[2], transform=transform)
    with rasterio.open(GHSL_MOSAIC_TIF, "w", **meta) as dst:
        dst.write(mosaic)
    for s in srcs:
        s.close()


def check_ghsl_covers_country(country: gpd.GeoDataFrame) -> bool:
    """Sanity check: does the merged GHSL mosaic bounding box contain the PH boundary?"""
    build_ghsl_mosaic()
    with rasterio.open(GHSL_MOSAIC_TIF) as src:
        left, bottom, right, top = src.bounds
    minx, miny, maxx, maxy = country.total_bounds
    covers = (left <= minx) and (bottom <= miny) and (right >= maxx) and (top >= maxy)
    if not covers:
        print(
            "[WARNING] GHSL mosaic does not fully cover the PH boundary.\n"
            f"  mosaic bounds : {(left, bottom, right, top)}\n"
            f"  country bounds: {(minx, miny, maxx, maxy)}\n"
            "  Candidate cells outside the mosaic will have no built-up value "
            "and will be marked ineligible by default -- check for a missing "
            "GHSL tile (e.g. Batanes, or Sulu/Tawi-Tawi)."
        )
    return covers


def load_stations() -> gpd.GeoDataFrame:
    df = pd.read_csv(STATION_LIST_CSV)
    missing = [c for c in (STATION_LON_COL, STATION_LAT_COL) if c not in df.columns]
    if missing:
        raise KeyError(
            f"Column(s) {missing} not found in {STATION_LIST_CSV.name}.\n"
            f"Available columns: {list(df.columns)}\n"
            f"Update STATION_LAT_COL / STATION_LON_COL in the Config cell above."
        )
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df[STATION_LON_COL], df[STATION_LAT_COL]),
        crs=CRS_WGS84,
    )
    return gdf


## grid -- candidate monitoring-site grid, province + built-up attributes

In [ ]:
def make_candidate_grid(country: gpd.GeoDataFrame, cell_km: float = CANDIDATE_CELL_KM) -> gpd.GeoDataFrame:
    """Generates a regular grid of candidate points at `cell_km` spacing, clipped to the country boundary."""
    country_proj = country.to_crs(CRS_PROJECTED)
    minx, miny, maxx, maxy = country_proj.total_bounds
    step_m = cell_km * 1000

    xs = np.arange(minx, maxx, step_m)
    ys = np.arange(miny, maxy, step_m)
    xx, yy = np.meshgrid(xs, ys)
    pts = gpd.GeoSeries([Point(x, y) for x, y in zip(xx.ravel(), yy.ravel())], crs=CRS_PROJECTED)

    grid = gpd.GeoDataFrame(geometry=pts)
    union = country_proj.geometry.union_all()
    grid = grid[grid.within(union)].reset_index(drop=True)
    grid["candidate_id"] = grid.index

    return grid.to_crs(CRS_WGS84)


def attach_province(points: gpd.GeoDataFrame, provinces: gpd.GeoDataFrame,
                     province_col: str = "province") -> gpd.GeoDataFrame:
    """
    Generic spatial join: tags any point GeoDataFrame (candidate grid OR
    population points) with the province it falls in. Points that don't land
    exactly inside a polygon fall back to nearest-province matching.
    """
    original_index = points.index

    joined = gpd.sjoin(points, provinces[[ADMIN2_NAME_FIELD, "geometry"]],
                        how="left", predicate="within")
    joined = joined.rename(columns={ADMIN2_NAME_FIELD: province_col}).drop(columns=["index_right"])
    joined = joined[~joined.index.duplicated(keep="first")]
    joined = joined.reindex(original_index)

    missing = joined[province_col].isna()
    if missing.any():
        points_proj = points.loc[missing, ["geometry"]].to_crs(CRS_PROJECTED)
        provinces_proj = provinces[[ADMIN2_NAME_FIELD, "geometry"]].to_crs(CRS_PROJECTED)

        nearest = gpd.sjoin_nearest(points_proj, provinces_proj, how="left")
        nearest = nearest[~nearest.index.duplicated(keep="first")]
        nearest = nearest.reindex(points_proj.index)

        joined.loc[missing, province_col] = nearest[ADMIN2_NAME_FIELD]

    return joined


def attach_builtup(grid: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Samples the GHSL built-up fraction at each candidate point location."""
    with rasterio.open(GHSL_MOSAIC_TIF) as src:
        coords = [(p.x, p.y) for p in grid.geometry]
        values = np.array([v[0] for v in src.sample(coords)], dtype=float)

    grid = grid.copy()
    grid["builtup_frac"] = values
    grid["is_eligible"] = grid["builtup_frac"] >= BUILTUP_THRESHOLD
    return grid


## coverage -- spatial coverage relationships (BallTree / haversine)

In [ ]:
EARTH_RADIUS_KM = 6371.0088  # mean Earth radius (IUGG)


def _to_radians(gdf: gpd.GeoDataFrame) -> np.ndarray:
    lat = gdf.geometry.y.to_numpy()
    lon = gdf.geometry.x.to_numpy()
    return np.radians(np.column_stack([lat, lon]))


def mark_existing_coverage(pop_points: gpd.GeoDataFrame, stations: gpd.GeoDataFrame,
                            radius_km: float = RADIUS_KM) -> gpd.GeoDataFrame:
    """Flags each population point as covered (True) or not by the CURRENT network."""
    tree = BallTree(_to_radians(stations), metric="haversine")
    idx = tree.query_radius(_to_radians(pop_points), r=radius_km / EARTH_RADIUS_KM)
    pop_points = pop_points.copy()
    pop_points["covered_existing"] = [len(i) > 0 for i in idx]
    return pop_points


def drop_candidates_near_existing(grid: gpd.GeoDataFrame, stations: gpd.GeoDataFrame,
                                   min_dist_km: float = NO_OVERLAP_KM) -> gpd.GeoDataFrame:
    """Removes candidate sites within `min_dist_km` of any existing station."""
    tree = BallTree(_to_radians(stations), metric="haversine")
    dist, _ = tree.query(_to_radians(grid), k=1)
    dist_km = dist.ravel() * EARTH_RADIUS_KM
    grid = grid.copy()
    grid["dist_to_existing_km"] = dist_km
    return grid[dist_km >= min_dist_km].reset_index(drop=True)


def build_coverage_sets(grid: gpd.GeoDataFrame, pop_points: gpd.GeoDataFrame,
                         radius_km: float = RADIUS_KM):
    """
    Returns:
        uncovered : subset of pop_points NOT already covered by the existing
                    network (reset to a fresh 0..n-1 index -- this index is
                    what N_j's keys refer to)
        N_j       : dict {row position in `uncovered` -> [candidate_ids within radius_km]}
    """
    uncovered = pop_points[~pop_points["covered_existing"]].reset_index(drop=True)
    tree = BallTree(_to_radians(grid), metric="haversine")
    idx_lists = tree.query_radius(_to_radians(uncovered), r=radius_km / EARTH_RADIUS_KM)

    candidate_ids = grid["candidate_id"].to_numpy()
    N_j = {j: candidate_ids[idxs].tolist() for j, idxs in enumerate(idx_lists) if len(idxs) > 0}
    return uncovered, N_j


## optimize -- maximal-covering-location MILP (mirrors paper eq. 4-6)

In [ ]:
def solve_mclp(grid: pd.DataFrame, uncovered_pop: pd.DataFrame, N_j: dict,
                n_new: int, use_eligibility: bool = True,
                candidate_subset: Optional[Iterable[int]] = None,
                time_limit_sec: int = SOLVER_TIME_LIMIT_SEC):
    """
    Solves:
        maximize   sum_j w_j * z_j
        subject to sum_i x_i == n_new
                   z_j <= sum_{i in N_j} x_i      for every population point j
                   x_i == 0 for ineligible i      (only if use_eligibility)
                   x_i, z_j in {0, 1}

    Returns: chosen_ids, newly_covered_pop, status
    """
    cand_df = grid if candidate_subset is None else grid[grid["candidate_id"].isin(candidate_subset)]
    candidate_ids = cand_df["candidate_id"].tolist()

    if n_new > len(candidate_ids):
        print(
            f"[WARNING] solve_mclp: requested n_new={n_new} but only "
            f"{len(candidate_ids)} candidate sites are available in this subset "
            f"-- capping to {len(candidate_ids)}. If this is unexpected, check "
            f"that province budgets were computed with the correct "
            f"max_per_province capacity for this eligibility setting."
        )
        n_new = len(candidate_ids)

    prob = pulp.LpProblem("mclp", pulp.LpMaximize)
    x = {i: pulp.LpVariable(f"x_{i}", cat="Binary") for i in candidate_ids}

    relevant_j = {j: [i for i in ids if i in x] for j, ids in N_j.items()}
    relevant_j = {j: ids for j, ids in relevant_j.items() if ids}
    z = {j: pulp.LpVariable(f"z_{j}", cat="Binary") for j in relevant_j}

    w = uncovered_pop["population"].to_dict()
    prob += pulp.lpSum(w[j] * z[j] for j in z)               # objective (eq. 4)
    prob += pulp.lpSum(x.values()) == n_new                  # station budget (eq. 5)

    for j, ids in relevant_j.items():
        prob += z[j] <= pulp.lpSum(x[i] for i in ids)        # coverage link

    if use_eligibility:
        elig = cand_df.set_index("candidate_id")["is_eligible"]
        for i in candidate_ids:
            if not bool(elig.get(i, False)):
                prob += x[i] == 0                            # impervious constraint

    solver = pulp.PULP_CBC_CMD(msg=False, timeLimit=time_limit_sec)
    prob.solve(solver)

    chosen_ids = [i for i, var in x.items() if pulp.value(var) is not None and pulp.value(var) > 0.5]
    newly_covered = sum(w[j] for j in z if pulp.value(z[j]) is not None and pulp.value(z[j]) > 0.5)
    return chosen_ids, newly_covered, pulp.LpStatus[prob.status]


## allocation -- province budgets (capacity-aware, with feasibility guards)

In [ ]:
def compute_province_capacity(grid: pd.DataFrame, use_eligibility: bool,
                               province_col: str = "province") -> dict:
    """
    Counts available candidate sites per province, matching the same scoping
    a province-level solve_mclp() call would see: eligible-only sites when
    use_eligibility=True (Model 2), all sites when False (Model 3).
    """
    df = grid[grid["is_eligible"]] if use_eligibility else grid
    return df.groupby(province_col)["candidate_id"].nunique().to_dict()


def compute_province_budgets(uncovered_pop_with_province: pd.DataFrame,
                              total_new_stations: int = N_NEW_STATIONS,
                              min_per_province: int = MIN_STATIONS_PER_PROVINCE,
                              max_per_province: Optional[dict] = None) -> dict:
    """
    Splits total_new_stations across provinces proportional to each
    province's share of currently-uncovered population.

    max_per_province : optional dict {province_name: max_n_new_stations}, e.g.
        the count of available (eligible) candidate sites in that province.
        Provinces absent from this dict are treated as uncapped.

    Raises ValueError up front if total_new_stations can't be reconciled with
    min_per_province or max_per_province, instead of looping forever.
    """
    prov_pop = uncovered_pop_with_province.groupby("province")["population"].sum()
    provinces = prov_pop.index.tolist()
    n_provinces = len(provinces)

    floor_total = min_per_province * n_provinces
    if total_new_stations < floor_total:
        raise ValueError(
            f"total_new_stations ({total_new_stations}) is less than "
            f"min_per_province ({min_per_province}) x n_provinces ({n_provinces}) "
            f"= {floor_total}. Raise N_NEW_STATIONS, lower MIN_STATIONS_PER_PROVINCE, "
            f"or reduce the number of provinces in scope."
        )

    if max_per_province is not None:
        total_capacity = sum(max_per_province.get(p, float("inf")) for p in provinces)
        if total_new_stations > total_capacity:
            raise ValueError(
                f"total_new_stations ({total_new_stations}) exceeds total candidate "
                f"capacity across provinces ({total_capacity:,.0f}). Lower N_NEW_STATIONS "
                f"or widen the candidate grid (e.g. smaller CANDIDATE_CELL_KM)."
            )
        capacity_floor = sum(min(max_per_province.get(p, float("inf")), min_per_province)
                              for p in provinces)
        if capacity_floor < floor_total:
            short = [p for p in provinces
                     if max_per_province.get(p, float("inf")) < min_per_province]
            raise ValueError(
                f"min_per_province ({min_per_province}) exceeds available candidate "
                f"capacity in: {short}. Lower MIN_STATIONS_PER_PROVINCE or expand the "
                f"candidate grid for those provinces."
            )

    shares = prov_pop / prov_pop.sum()

    budgets = (shares * total_new_stations).round().astype(int)
    budgets = budgets.clip(lower=min_per_province)

    if max_per_province is not None:
        for p in provinces:
            cap = max_per_province.get(p)
            if cap is not None:
                budgets[p] = min(budgets[p], cap)

    # Rounding (and capping) can drift the total off target; nudge the
    # largest-share provinces up/down by 1 until it matches exactly.
    diff = total_new_stations - budgets.sum()
    if diff != 0:
        order = shares.sort_values(ascending=False).index.tolist()
        i = 0
        stall_guard = 0
        max_stalls = len(order) + 1
        while diff != 0:
            prov = order[i % len(order)]
            step = 1 if diff > 0 else -1

            blocked = False
            if step < 0 and budgets[prov] <= min_per_province:
                blocked = True
            if step > 0 and max_per_province is not None:
                cap = max_per_province.get(prov)
                if cap is not None and budgets[prov] >= cap:
                    blocked = True

            if blocked:
                i += 1
                stall_guard += 1
                if stall_guard > max_stalls:
                    raise RuntimeError(
                        "compute_province_budgets: could not converge on a feasible "
                        "allocation despite passing feasibility checks -- this is a bug, "
                        "please report the inputs that triggered it."
                    )
                continue

            budgets[prov] += step
            diff -= step
            i += 1
            stall_guard = 0

    return budgets.to_dict()


## metrics -- PCR before/after, MARR, model comparison table

In [ ]:
def pcr_before(pop_points: pd.DataFrame) -> float:
    total = pop_points["population"].sum()
    covered = pop_points.loc[pop_points["covered_existing"], "population"].sum()
    return covered / total


def pcr_after(pop_points: pd.DataFrame, newly_covered_population: float) -> float:
    total = pop_points["population"].sum()
    covered_before = pop_points.loc[pop_points["covered_existing"], "population"].sum()
    return (covered_before + newly_covered_population) / total


def compute_marr(pop_points: gpd.GeoDataFrame, all_stations: gpd.GeoDataFrame,
                  radius_km: float = RADIUS_KM) -> float:
    """
    Monitoring Area Repetition Rate (MARR): population-weighted average number
    of REDUNDANT stations covering an already-covered population point, using
    the full network (existing + newly placed stations for this model).

        MARR = sum_p[ pop_p * (n_stations_covering_p - 1) ] / sum_p[ pop_p ]
               over points p covered by >= 1 station

    0.0 -> every covered point is served by exactly one station (no overlap)
    1.0 -> covered points are served by two stations on average
    etc.

    WORKING DEFINITION -- MARR is not otherwise specified in this codebase.
    This mirrors pcr_after's convention of scoring the combined existing+new
    network, population-weighted like the rest of this section. Confirm
    against your source formula and adjust this function if the definition
    differs -- everything that calls it only depends on the float it returns.
    """
    if len(all_stations) == 0 or len(pop_points) == 0:
        return 0.0

    tree = BallTree(_to_radians(all_stations), metric="haversine")
    idx_lists = tree.query_radius(
        _to_radians(pop_points), r=radius_km / EARTH_RADIUS_KM
    )
    counts = np.array([len(i) for i in idx_lists])

    covered_mask = counts >= 1
    pop = pop_points["population"].to_numpy()
    covered_pop = pop[covered_mask]
    covered_counts = counts[covered_mask]

    total_covered_pop = covered_pop.sum()
    if total_covered_pop == 0:
        return 0.0

    return float(np.sum(covered_pop * (covered_counts - 1)) / total_covered_pop)


def summarize_models(results: dict, pop_points: pd.DataFrame,
                      marr_values: Optional[dict] = None) -> pd.DataFrame:
    """
    results     : {model_name: {"chosen_ids": [...], "newly_covered_pop": float}}
    marr_values : optional {model_name: float}, from compute_marr() per model.
    """
    base_pcr = pcr_before(pop_points)
    rows = []
    for name, r in results.items():
        row = {
            "model": name,
            "n_stations_placed": len(r["chosen_ids"]),
            "newly_covered_population": r["newly_covered_pop"],
            "pcr_before": base_pcr,
            "pcr_after": pcr_after(pop_points, r["newly_covered_pop"]),
            "pcr_gain_pp": (pcr_after(pop_points, r["newly_covered_pop"]) - base_pcr) * 100,
        }
        if marr_values is not None:
            row["marr"] = marr_values.get(name)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("pcr_after", ascending=False).reset_index(drop=True)


## visualize -- static maps of chosen sites vs existing network (matplotlib + geopandas)

In [ ]:
def _site_points(chosen_df: pd.DataFrame) -> gpd.GeoDataFrame:
    """Builds a point GeoDataFrame from a chosen-sites table's lon/lat columns."""
    return gpd.GeoDataFrame(
        chosen_df,
        geometry=gpd.points_from_xy(chosen_df["lon"], chosen_df["lat"]),
        crs=CRS_WGS84,
    )


def plot_model_sites(model_name: str, chosen_df: pd.DataFrame,
                      country: gpd.GeoDataFrame, stations: gpd.GeoDataFrame,
                      out_path) -> None:
    """Saves one PNG: country outline + existing stations + this model's new sites."""
    sites = _site_points(chosen_df)

    fig, ax = plt.subplots(figsize=(8, 10))
    country.boundary.plot(ax=ax, color="black", linewidth=0.5)
    stations.plot(ax=ax, color="gray", markersize=6, label=f"Existing ({len(stations)})")
    sites.plot(ax=ax, color="crimson", markersize=8, label=f"New ({len(sites)})")
    ax.set_title(model_name, fontsize=11)
    ax.set_axis_off()
    ax.legend(loc="lower left", fontsize=8, frameon=True)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_all_models(results: dict, grid_df: pd.DataFrame, country: gpd.GeoDataFrame,
                     stations: gpd.GeoDataFrame, out_path) -> None:
    """Saves one PNG with a 2x2 panel comparing every model's chosen sites side by side."""
    n = len(results)
    ncols = 2
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 8 * nrows))
    axes = axes.ravel() if n > 1 else [axes]

    for ax, (name, r) in zip(axes, results.items()):
        chosen_df = grid_df[grid_df["candidate_id"].isin(r["chosen_ids"])]
        sites = _site_points(chosen_df)
        country.boundary.plot(ax=ax, color="black", linewidth=0.4)
        stations.plot(ax=ax, color="gray", markersize=3)
        sites.plot(ax=ax, color="crimson", markersize=4)
        ax.set_title(f"{name}\n({len(sites)} new stations)", fontsize=9)
        ax.set_axis_off()

    for ax in axes[n:]:  # hide any unused panels if results count is odd
        ax.set_visible(False)

    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


## check_inputs (optional)

Run this section before the main pipeline the first time, or after editing
any config constant above, to confirm file paths and column names resolve
correctly. Safe to skip on later runs.

In [ ]:
def check_admin2():
    print("=" * 70)
    print(f"phl_admin2.shp  ->  {ADMIN2_SHP}")
    gdf = gpd.read_file(ADMIN2_SHP)
    print(f"  rows: {len(gdf)}")
    print(f"  columns: {list(gdf.columns)}")
    print(f"  CRS: {gdf.crs}")
    print("  first 3 rows (non-geometry columns):")
    print(gdf.drop(columns="geometry").head(3).to_string())
    print(f"\n  ==> current ADMIN2_NAME_FIELD = '{ADMIN2_NAME_FIELD}' "
          f"{'FOUND' if ADMIN2_NAME_FIELD in gdf.columns else '*** NOT FOUND ***'}")


def check_population_csv():
    print("=" * 70)
    print(f"population CSV  ->  {POP_DENSITY_CSV}")
    df = pd.read_csv(POP_DENSITY_CSV, nrows=5)
    print(f"  columns: {list(df.columns)}")
    print("  first 5 rows:")
    print(df.to_string())
    for key, col in POP_CSV_COLS.items():
        status = "FOUND" if col in df.columns else "*** NOT FOUND ***"
        print(f"  ==> current POP_CSV_COLS['{key}'] = '{col}'  {status}")


def check_stations():
    print("=" * 70)
    print(f"station list  ->  {STATION_LIST_CSV}")
    df = pd.read_csv(STATION_LIST_CSV, nrows=5)
    print(f"  columns: {list(df.columns)}")
    print("  first 5 rows:")
    print(df.to_string())
    for key, col in [("STATION_LAT_COL", STATION_LAT_COL), ("STATION_LON_COL", STATION_LON_COL)]:
        status = "FOUND" if col in df.columns else "*** NOT FOUND ***"
        print(f"  ==> current {key} = '{col}'  {status}")


def check_admin0():
    print("=" * 70)
    print(f"phl_admin0.shp  ->  {ADMIN0_SHP}")
    gdf = gpd.read_file(ADMIN0_SHP)
    print(f"  rows: {len(gdf)}, CRS: {gdf.crs}")
    print(f"  bounds: {gdf.total_bounds}")


def check_ghsl_tiles():
    print("=" * 70)
    print("GHSL built-up tiles:")
    for p in GHSL_TILES:
        exists = p.exists()
        print(f"  {p.name}  exists={exists}")
        if exists:
            with rasterio.open(p) as src:
                print(f"    bounds={src.bounds}, crs={src.crs}, shape={src.shape}")


def run_input_checks():
    check_admin0()
    check_admin2()
    check_population_csv()
    check_stations()
    check_ghsl_tiles()
    print("=" * 70)
    print("Done. Fix any '*** NOT FOUND ***' lines above by editing the "
          "matching constant in the Config cell, then re-run this cell to "
          "confirm before moving on to the main pipeline below.")


In [ ]:
# Uncomment to run input checks (recommended on first run, or after editing config):
# run_input_checks()


## run_models -- main pipeline

Set `REBUILD_ABT = True` below to force rebuilding the candidate grid/ABT
even if a cached parquet exists (needed after changing `CANDIDATE_CELL_KM` or
`NO_OVERLAP_KM` in the Config cell). `BUILTUP_THRESHOLD` is always safe to
change without a rebuild -- eligibility is recomputed from cached
`builtup_frac` every time.

In [ ]:
REBUILD_ABT = False  # set True to force a fresh grid/ABT build


def _fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    return f"{seconds / 60:.1f}min"


class _Stage:
    """Small context manager that prints how long a pipeline stage took."""
    def __init__(self, label: str):
        self.label = label

    def __enter__(self):
        print(f"{self.label}...")
        self._t0 = time.time()
        return self

    def __exit__(self, exc_type, exc, tb):
        if exc_type is None:
            print(f"  done in {_fmt_elapsed(time.time() - self._t0)}")


def build_abt(force_rebuild: bool = False):
    """
    Builds (or loads a cached) analytical base table: the candidate grid with
    province, built-up eligibility, and distance-to-existing-station attached.
    Cached as GeoParquet at ABT_PARQUET after the first build.
    """
    provinces = load_provinces()
    stations = load_stations()

    if ABT_PARQUET.exists() and not force_rebuild:
        print(f"Loading cached ABT from {ABT_PARQUET} (set REBUILD_ABT = True to force a rebuild)")
        grid = gpd.read_parquet(ABT_PARQUET)
        grid["is_eligible"] = grid["builtup_frac"] >= BUILTUP_THRESHOLD
        return grid, stations, provinces

    print("No cached ABT found (or rebuild forced) -- building candidate grid from scratch...")
    country = load_country_boundary()
    check_ghsl_covers_country(country)

    grid = make_candidate_grid(country)
    grid = attach_province(grid, provinces)
    grid = attach_builtup(grid)
    grid = drop_candidates_near_existing(grid, stations)

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    grid.to_parquet(ABT_PARQUET)  # GeoParquet -- keeps geometry + CRS
    return grid, stations, provinces


def prepare_population(stations: gpd.GeoDataFrame, provinces: gpd.GeoDataFrame):
    pop_points = load_population_points()
    pop_points = mark_existing_coverage(pop_points, stations)
    pop_points = attach_province(pop_points, provinces)
    return pop_points


def run_model_1_national_modified(grid, uncovered_pop, N_j):
    chosen, new_pop, status = solve_mclp(
        grid, uncovered_pop, N_j, n_new=N_NEW_STATIONS, use_eligibility=True
    )
    print(f"[Model 1] status={status}, stations={len(chosen)}, newly_covered_pop={new_pop:,.0f}")
    return {"chosen_ids": chosen, "newly_covered_pop": new_pop}


def run_model_4_national_simple(grid, uncovered_pop, N_j):
    chosen, new_pop, status = solve_mclp(
        grid, uncovered_pop, N_j, n_new=N_NEW_STATIONS, use_eligibility=False
    )
    print(f"[Model 4] status={status}, stations={len(chosen)}, newly_covered_pop={new_pop:,.0f}")
    return {"chosen_ids": chosen, "newly_covered_pop": new_pop}


def run_model_by_province(grid, uncovered_pop, N_j, province_budgets: dict,
                           use_eligibility: bool, label: str):
    all_chosen, total_new_pop = [], 0.0
    for province, n_new in tqdm(province_budgets.items(), total=len(province_budgets),
                                 desc=f"[{label}] provinces", unit="province"):
        subset_ids = grid.loc[grid["province"] == province, "candidate_id"]
        if n_new <= 0 or subset_ids.empty:
            continue
        chosen, new_pop, status = solve_mclp(
            grid, uncovered_pop, N_j, n_new=n_new,
            use_eligibility=use_eligibility, candidate_subset=subset_ids,
        )
        if status != "Optimal":
            print(f"  [{label}] {province}: status={status} (n_new={n_new})")
        all_chosen.extend(chosen)
        total_new_pop += new_pop
    print(f"[{label}] stations={len(all_chosen)}, newly_covered_pop={total_new_pop:,.0f}")
    return {"chosen_ids": all_chosen, "newly_covered_pop": total_new_pop}


### Step 1 -- Build ABT (candidate grid, province tags, built-up eligibility)

In [ ]:
run_start = time.time()

with _Stage("Building ABT (candidate grid, province tags, built-up eligibility)"):
    grid, stations, provinces = build_abt(force_rebuild=REBUILD_ABT)
    # Keep lon/lat as plain columns before dropping geometry -- otherwise every
    # downstream CSV (including the chosen-site tables) loses the coordinates
    # entirely, since geometry is what actually carries them.
    grid_df = pd.DataFrame(
        grid.assign(lon=grid.geometry.x, lat=grid.geometry.y).drop(columns="geometry")
    )

grid_df.head()


### Step 2 -- Load population points, mark existing coverage

In [ ]:
with _Stage("Loading population points and marking existing coverage"):
    pop_points = prepare_population(stations, provinces)

pop_points.head()


### Step 3 -- Build candidate <-> population coverage sets

In [ ]:
with _Stage("Building candidate<->population coverage sets"):
    uncovered_pop, N_j = build_coverage_sets(grid, pop_points)
    uncovered_pop_df = pd.DataFrame(uncovered_pop.drop(columns="geometry"))

uncovered_pop_df.head()


### Step 4 -- Province budgets

Model 2 (eligibility-filtered) and Model 3 (unfiltered) see different candidate pools per province, so each gets its own capacity-aware budget.

In [ ]:
with _Stage("Computing province budgets"):
    capacity_modified = compute_province_capacity(grid_df, use_eligibility=True)
    capacity_simple = compute_province_capacity(grid_df, use_eligibility=False)

    province_budgets_modified = compute_province_budgets(
        uncovered_pop_df, max_per_province=capacity_modified
    )
    province_budgets_simple = compute_province_budgets(
        uncovered_pop_df, max_per_province=capacity_simple
    )


### Step 5 -- Run all four models

In [ ]:
results = {}
with _Stage("[Model 1] solving (national, modified framework)"):
    results["Model 1: National + Modified Framework"] = run_model_1_national_modified(
        grid_df, uncovered_pop_df, N_j
    )
with _Stage("[Model 2] solving (per-province, modified framework)"):
    results["Model 2: Province + Modified Framework"] = run_model_by_province(
        grid_df, uncovered_pop_df, N_j, province_budgets_modified,
        use_eligibility=True, label="Model 2",
    )
with _Stage("[Model 3] solving (per-province, simple distance)"):
    results["Model 3: Province + Simple Distance"] = run_model_by_province(
        grid_df, uncovered_pop_df, N_j, province_budgets_simple,
        use_eligibility=False, label="Model 3",
    )
with _Stage("[Model 4] solving (national, simple distance)"):
    results["Model 4: National + Simple Distance"] = run_model_4_national_simple(
        grid_df, uncovered_pop_df, N_j
    )


### Step 6 -- Compute MARR per model

Uses the full network (existing + this model's new sites) for each model.

In [ ]:
with _Stage("Computing MARR (Monitoring Area Repetition Rate) per model"):
    marr_values = {}
    for name, r in results.items():
        chosen_df = grid_df[grid_df["candidate_id"].isin(r["chosen_ids"])]
        new_sites = gpd.GeoDataFrame(
            chosen_df,
            geometry=gpd.points_from_xy(chosen_df["lon"], chosen_df["lat"]),
            crs=CRS_WGS84,
        )
        all_stations_for_model = gpd.GeoDataFrame(
            pd.concat([stations[["geometry"]], new_sites[["geometry"]]], ignore_index=True),
            geometry="geometry", crs=CRS_WGS84,
        )
        marr_values[name] = compute_marr(pop_points, all_stations_for_model)

marr_values


### Step 7 -- Summarize, write results and visualizations

Outputs land in `data/results/n<N_NEW_STATIONS>/`, namespaced by station count.

In [ ]:
summary = summarize_models(results, pop_points, marr_values=marr_values)

run_tag = f"n{N_NEW_STATIONS}"
run_results_dir = RESULTS_DIR / run_tag
run_results_dir.mkdir(parents=True, exist_ok=True)

summary.to_csv(run_results_dir / f"model_comparison_{run_tag}.csv", index=False)

with _Stage("Writing chosen-site tables and map visualizations"):
    country = load_country_boundary()
    for name, r in results.items():
        chosen_df = grid_df[grid_df["candidate_id"].isin(r["chosen_ids"])]
        safe_name = name.split(":")[0].replace(" ", "_").lower()
        chosen_df.to_csv(run_results_dir / f"{safe_name}_{run_tag}_sites.csv", index=False)
        plot_model_sites(
            name, chosen_df, country, stations,
            out_path=run_results_dir / f"{safe_name}_{run_tag}_map.png",
        )
    plot_all_models(
        results, grid_df, country, stations,
        out_path=run_results_dir / f"all_models_{run_tag}_comparison.png",
    )

print("\n=== Model comparison ===")
print(summary.to_string(index=False))
print(f"\nWritten to {run_results_dir}")
print(f"Total runtime: {_fmt_elapsed(time.time() - run_start)}")

summary
